In [1]:
import pandas as pd
import requests
import json
import time
import datetime

In [2]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

In [3]:
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

,city
0,Mont Saint Michel
1,St Malo
2,Bayeux
3,Le Havre
4,Rouen


In [4]:
df = df_source.copy()
for index, row in df.iterrows():
    print(index, row["city"])
    res = requests.get(f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json", headers=headers)
    city = res.json()[0]
    df.loc[index, "lat"] = city["lat"]
    df.loc[index, "lon"] = city["lon"]
    time.sleep(1)
df.head()
df.to_csv("cities_with_geoposition.csv", index=False)

0 Mont Saint Michel
1 St Malo
2 Bayeux
3 Le Havre
4 Rouen
5 Paris
6 Amiens
7 Lille
8 Strasbourg
9 Chateau du Haut Koenigsbourg
10 Colmar
11 Eguisheim
12 Besancon
13 Dijon
14 Annecy
15 Grenoble
16 Lyon
17 Gorges du Verdon
18 Bormes les Mimosas
19 Cassis
20 Marseille
21 Aix en Provence
22 Avignon
23 Uzes
24 Nimes
25 Aigues Mortes
26 Saintes Maries de la mer
27 Collioure
28 Carcassonne
29 Ariege
30 Toulouse
31 Montauban
32 Biarritz
33 Bayonne
34 La Rochelle


In [5]:
df = pd.read_csv("cities_with_geoposition.csv")
df.head()

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966


In [10]:
list_weather_data = []

for index, row in df.iterrows():
    res_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid=4656ed7337e689999007412af6a4dafe", headers=headers)
    res_weather_json = res_weather.json()
    
    for res in res_weather_json['list']:
        weather_entry = {
            "city": row['city'],
            "lat": row['lat'],
            "lon": row['lon'],
            "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%d/%m/%Y'),
            "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
            "temp": res['main']['temp'],
            "prob_rain": res['pop'],
            "volume_rain": res.get('rain', {}).get('3h', 0),
            "wind_speed": res['wind']['speed'],
            "perc_cloud": res['clouds']['all']
        }
        list_weather_data.append(weather_entry)
    time.sleep(1)

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())
df_weather.to_csv("weather_forecast.csv", index=False)

                city        lat      lon        date   hour   temp  prob_rain  \
0  Mont Saint Michel  48.635954 -1.51146  09/02/2026  22:00   8.97       0.84   
1  Mont Saint Michel  48.635954 -1.51146  10/02/2026  01:00   9.18       0.99   
2  Mont Saint Michel  48.635954 -1.51146  10/02/2026  04:00   9.46       0.40   
3  Mont Saint Michel  48.635954 -1.51146  10/02/2026  07:00   9.78       0.01   
4  Mont Saint Michel  48.635954 -1.51146  10/02/2026  10:00  10.19       0.00   

   volume_rain  wind_speed  perc_cloud  
0         0.47        4.32          76  
1         1.32        5.63          80  
2         0.30        5.99          87  
3         0.00        4.88          95  
4         0.00        5.08         100  


In [11]:
df_weather = pd.read_csv("weather_forecast.csv")
df_weather.head()

,city,lat,lon,date,hour,temp,prob_rain,volume_rain,wind_speed,perc_cloud
0,Mont Saint Michel,48.635954,-1.51146,09/02/2026,22:00,8.97,0.84,0.47,4.32,76
1,Mont Saint Michel,48.635954,-1.51146,10/02/2026,01:00,9.18,0.99,1.32,5.63,80
2,Mont Saint Michel,48.635954,-1.51146,10/02/2026,04:00,9.46,0.40,0.30,5.99,87
3,Mont Saint Michel,48.635954,-1.51146,10/02/2026,07:00,9.78,0.01,0.00,4.88,95
4,Mont Saint Michel,48.635954,-1.51146,10/02/2026,10:00,10.19,0.00,0.00,5.08,100


In [12]:
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

,city,lat,lon,date,hour,temp,prob_rain,volume_rain,wind_speed,perc_cloud
0,Mont Saint Michel,48.635954,-1.51146,09/02/2026,22:00,8.97,84.0,0.47,15.552,76
1,Mont Saint Michel,48.635954,-1.51146,10/02/2026,01:00,9.18,99.0,1.32,20.268,80
2,Mont Saint Michel,48.635954,-1.51146,10/02/2026,04:00,9.46,40.0,0.30,21.564,87
3,Mont Saint Michel,48.635954,-1.51146,10/02/2026,07:00,9.78,1.0,0.00,17.568,95
4,Mont Saint Michel,48.635954,-1.51146,10/02/2026,10:00,10.19,0.0,0.00,18.288,100


In [13]:
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({'temp': ['mean', 'min', 'max'], 'prob_rain': 'max', 'volume_rain': {'mean', 'max', 'sum'}, 'wind_speed': 'max', 'perc_cloud': 'mean'}).reset_index()
df_weather_groupby.head()

city        lat      lon        date    temp                \
                                                    mean    min    max   
0  Aigues Mortes  43.566152  4.19154  09/02/2026  10.720  10.72  10.72   
1  Aigues Mortes  43.566152  4.19154  10/02/2026  10.755   9.86  11.61   
2  Aigues Mortes  43.566152  4.19154  11/02/2026  13.515  11.87  15.12   
3  Aigues Mortes  43.566152  4.19154  12/02/2026  12.095  10.95  13.20   
4  Aigues Mortes  43.566152  4.19154  13/02/2026  10.650   9.39  12.14   

  prob_rain volume_rain                 wind_speed perc_cloud  
        max         max    sum     mean        max       mean  
0       0.0        0.00   0.00  0.00000     11.016    100.000  
1     100.0        3.22   8.84  1.10500     28.620     95.625  
2     100.0        1.16   1.29  0.16125     35.244     68.500  
3     100.0        2.00   3.05  0.38125     45.108     86.250  
4     100.0       11.31  15.69  1.96125     34.092     79.875

In [14]:
# --- DEFINITION DU SCORE METEO ---

# 1. Création des scores unitaires (Normalisation sur 100)
# Objectif : 100 = Parfait, 0 = Horrible

# Température (Cible 25°C) : On perd 4 pts par degré d'écart
# .clip(lower=0) empêche d'avoir des notes négatives
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Pluie Probabilité : 0% = 100 pts. 100% = 0 pts.
# (Note : prob_rain est entre 0 et 1, donc x100 pour le mettre en %)
df_weather_groupby['score_rain_prob'] = 100 - df_weather_groupby[('prob_rain', 'max')]

# Pluie Volume : 0mm = 100 pts. On perd 5 pts par mm.
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Vent (km/h) : 0 km/h = 100 pts. On perd 1 pt par km/h.
# Attention : wind_speed est en m/s, donc on convertit en km/h (* 3.6)
df_weather_groupby['score_wind'] = 100 - (df_weather_groupby[('wind_speed', 'max')] * 3.6)
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Nuages : 0% = 100 pts.
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Score Final Pondéré
# Poids : Temp(30%), PluieProba(20%), PluieVol(30%), Vent(10%), Nuages(10%)
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. Afficher le TOP 5 des villes (Moyenne sur la période)
top_cities = df_weather_groupby.groupby('city')['total_score'].mean().sort_values(ascending=False)
print("--- CLASSEMENT FINAL ---")
print(top_cities.head(10))

# Afficher les détails
df_weather_groupby.head()

--- CLASSEMENT FINAL ---
city
Collioure             52.059324
Avignon               48.793673
Marseille             48.097676
Bormes les Mimosas    47.808713
Gorges du Verdon      47.620802
Nimes                 46.908133
Aix en Provence       46.810450
Lyon                  46.130053
Uzes                  45.515249
Cassis                45.120193
Name: total_score, dtype: float64


city        lat      lon        date    temp                \
                                                    mean    min    max   
0  Aigues Mortes  43.566152  4.19154  09/02/2026  10.720  10.72  10.72   
1  Aigues Mortes  43.566152  4.19154  10/02/2026  10.755   9.86  11.61   
2  Aigues Mortes  43.566152  4.19154  11/02/2026  13.515  11.87  15.12   
3  Aigues Mortes  43.566152  4.19154  12/02/2026  12.095  10.95  13.20   
4  Aigues Mortes  43.566152  4.19154  13/02/2026  10.650   9.39  12.14   

  prob_rain volume_rain                 wind_speed perc_cloud score_temp  \
        max         max    sum     mean        max       mean              
0       0.0        0.00   0.00  0.00000     11.016    100.000      42.88   
1     100.0        3.22   8.84  1.10500     28.620     95.625      46.44   
2     100.0        1.16   1.29  0.16125     35.244     68.500      60.48   
3     100.0        2.00   3.05  0.38125     45.108     86.250      52.80   
4     100.0       11.31  15.69  1.96125     34.092     79.875      48.56   

  score_rain_prob score_rain_vol score_wind score_cloud total_score  
                                                                     
0           100.0         100.00    60.3424       0.000    68.89824  
1             0.0          55.80     0.0000       4.375    31.10950  
2             0.0          93.55     0.0000      31.500    49.35900  
3             0.0          84.75     0.0000      13.750    42.64000  
4             0.0          21.55     0.0000      20.125    23.04550

In [15]:
df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)